In [ ]:
"""Train one player recommender model per position group.

This script trains separate machine learning models for different football
position groups. Each model learns player performance profiles and can later
be used to recommend similar players within the same position group.
"""

import sys
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

CURRENT_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = CURRENT_DIR.parent
sys.path.append(str(PROJECT_ROOT))

from ml.toolkit.ml_utilities import (
    POSITION_GROUPS,
    RECOMMENDER_CATEGORICAL_COLUMNS,
    RECOMMENDER_TARGET_COLUMN,
    build_preprocessor,
    evaluate_model,
    get_recommender_feature_columns,
    get_recommender_numeric_columns,
    prepare_features_and_target,
    print_evaluation,
    save_model,
    split_data_by_group,
    split_data_randomly,
)


DATASET_PATH = CURRENT_DIR / "recommender_model_dataset.csv"
MODEL_PATH_TEMPLATE = CURRENT_DIR / "recommender_model_{position_group}.pkl"

PLAYER_ID_COLUMN = "player_id"
POSITION_GROUP_COLUMN = "position_group"

RANDOM_STATE = 42
TEST_SIZE = 0.2
MIN_GROUP_ROWS_FOR_TEST = 8
MIN_GROUP_PLAYERS_FOR_GROUP_SPLIT = 3

MODEL_PARAMETERS = {
    "n_estimators": 500,
    "max_depth": 10,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}


def load_training_dataset(dataset_path: Path) -> pd.DataFrame:
    """
    Load the recommender training dataset from a CSV file.

    The function checks whether the dataset file exists and verifies that all
    required columns are available before returning the loaded DataFrame.
    """
    if not dataset_path.exists():
        raise FileNotFoundError(f"Training dataset not found: {dataset_path}")

    dataset = pd.read_csv(dataset_path)

    required_columns = {
        PLAYER_ID_COLUMN,
        POSITION_GROUP_COLUMN,
        RECOMMENDER_TARGET_COLUMN,
    }
    missing_columns = required_columns.difference(dataset.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns in training dataset: {sorted(missing_columns)}"
        )

    return dataset


def build_model_pipeline(
    numeric_columns: list[str],
    categorical_columns: list[str],
) -> Pipeline:
    """
    Build the machine learning pipeline for the recommender model.

    The pipeline first preprocesses numeric and categorical input features and
    then trains a Random Forest Regressor using the configured model parameters.
    """
    preprocessor = build_preprocessor(
        numeric_columns=numeric_columns,
        categorical_columns=categorical_columns,
        boolean_columns=[],
    )

    return Pipeline(
        steps=[
            ("preprocessing", preprocessor),
            ("model", RandomForestRegressor(**MODEL_PARAMETERS)),
        ]
    )


def split_group_data(
    group_dataset: pd.DataFrame,
    features: pd.DataFrame,
    target: pd.Series,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """
    Split a position-group dataset into training and test data.

    If the position group contains enough rows and enough unique players, the
    split is performed by player ID to avoid data leakage between training and
    test data. Otherwise, a regular random split is used.

    """
    enough_rows = len(group_dataset) >= MIN_GROUP_ROWS_FOR_TEST
    enough_players = (
        group_dataset[PLAYER_ID_COLUMN].nunique()
        >= MIN_GROUP_PLAYERS_FOR_GROUP_SPLIT
    )

    if enough_rows and enough_players:
        return split_data_by_group(
            dataframe=group_dataset,
            features=features,
            target=target,
            group_column=PLAYER_ID_COLUMN,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
        )

    return split_data_randomly(
        features=features,
        target=target,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )


def train_position_group_model(
    dataset: pd.DataFrame,
    position_group: str,
) -> dict[str, object] | None:
    """
    Train and save a recommender model for one position group.

    The function filters the dataset for a specific position group, prepares the
    features and target variable, trains a Random Forest model, evaluates its
    performance, saves the trained model, and returns a summary of the results.
    """
    group_dataset = dataset[dataset[POSITION_GROUP_COLUMN] == position_group].copy()
    group_dataset = group_dataset.dropna(subset=[RECOMMENDER_TARGET_COLUMN])

    if group_dataset.empty:
        print(f"Skipped {position_group}: no training rows available")
        return None

    feature_columns = get_recommender_feature_columns(group_dataset)
    numeric_columns = get_recommender_numeric_columns(group_dataset)
    categorical_columns = [
        column
        for column in RECOMMENDER_CATEGORICAL_COLUMNS
        if column in feature_columns
    ]

    features, target = prepare_features_and_target(
        dataframe=group_dataset,
        target_column=RECOMMENDER_TARGET_COLUMN,
        columns_to_drop=[PLAYER_ID_COLUMN, POSITION_GROUP_COLUMN],
    )

    features_train, features_test, target_train, target_test = split_group_data(
        group_dataset=group_dataset,
        features=features,
        target=target,
    )

    model = build_model_pipeline(
        numeric_columns=numeric_columns,
        categorical_columns=categorical_columns,
    )

    model.fit(features_train, target_train)

    predictions = model.predict(features_test)
    metrics = evaluate_model(target_test, predictions)

    print_evaluation(metrics, model_name=f"Recommender {position_group}")

    model.feature_columns_ = list(features.columns)
    model.numeric_columns_ = numeric_columns
    model.categorical_columns_ = categorical_columns
    model.position_group_ = position_group

    output_path = Path(str(MODEL_PATH_TEMPLATE).format(position_group=position_group))
    save_model(model, output_path)

    return {
        "position_group": position_group,
        "rows": len(group_dataset),
        "players": group_dataset[PLAYER_ID_COLUMN].nunique(),
        "mean_absolute_error": metrics["mean_absolute_error"],
        "root_mean_squared_error": metrics["root_mean_squared_error"],
        "r2_score": metrics["r2_score"],
        "model_path": str(output_path),
    }


def train_all_position_group_models(dataset: pd.DataFrame) -> pd.DataFrame:
    """
    Train recommender models for all configured position groups.

    The function iterates over all position groups, trains one model per group,
    collects the evaluation results, and returns them as a summary DataFrame.
    """
    results = []

    for position_group in POSITION_GROUPS:
        result = train_position_group_model(dataset, position_group)

        if result is not None:
            results.append(result)

    return pd.DataFrame(results)


def main() -> None:
    """
    Run the complete recommender model training process.

    The function loads the training dataset, trains recommender models for all
    configured position groups, and prints a summary of the training results.
    """
    dataset = load_training_dataset(DATASET_PATH)
    results = train_all_position_group_models(dataset)

    if not results.empty:
        print("\nTraining summary:")
        print(results.to_string(index=False))


if __name__ == "__main__":
    main()